# Short notebook 2 — Second selected submission (XGBoost)

**Course:** TDT4173 Modern Machine Learning in Practice — Course Project
**Group number:** `[TODO]`
**Kaggle team name:** `[TODO — must include the group number, e.g. "[34] A Fantastic Team"]`

| Full name | Student ID (from student card) |
|-----------|-------------------------------|
| `[TODO]`  | `[TODO]` |
| `[TODO]`  | `[TODO]` |
| `[TODO]`  | `[TODO]` |

This notebook reproduces our **second selected** Kaggle submission. It uses the **same features and
validation** as Short notebook 1 but a **different predictor family — XGBoost instead of LightGBM** —
so our two selected predictions are genuinely diverse (and the report can compare ≥2 model types).


## 1. Configuration & imports

In [1]:
# --- Configuration & imports -------------------------------------------------
# Only standard scientific-Python packages are used (see requirements.txt).
# No internet access and no external data are required to run this notebook.
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# The ONLY path you may need to change if the raw data is placed elsewhere.
DATA_DIR = Path("sintef_project/data")
KERNEL   = DATA_DIR / "kernel"
EXTENDED = DATA_DIR / "extended"

# Where intermediate artifacts (feature tables, models) may be cached at run time.
ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)

assert KERNEL.exists(), f"Raw data not found at {KERNEL.resolve()} - adjust DATA_DIR."
print("Data dir:", DATA_DIR.resolve())


Data dir: C:\Users\oscar\TDT4173\Eksamensoppgave\ML-group-project-TDT4173\sintef_project\data


## 2. Load raw data

In [2]:
# --- Load raw data (boilerplate) --------------------------------------------
price   = pd.read_csv(KERNEL / "Historical_day_ahead_price_2015_2025.csv", parse_dates=["date"])
inflow  = pd.read_csv(KERNEL / "Historical_inflow_1958_2025.csv", parse_dates=["date"])
volume  = pd.read_csv(KERNEL / "Historical_volume_2015_2024.csv", parse_dates=["date"])
wvalue  = pd.read_csv(KERNEL / "Synthetic_water_value_2015_2024.csv", parse_dates=["date"])
uc      = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv", parse_dates=["starttime"])
minflow = pd.read_csv(EXTENDED / "Constraint_min_flow.csv", parse_dates=["date"])
minvol  = pd.read_csv(EXTENDED / "Constraint_min_volume.csv", parse_dates=["date"])

# Test cases + required output layout
sample_sub = pd.read_csv("sintef_project/sample_submission.csv")
test_map   = pd.read_csv("sintef_project/prediction_mapping.csv", parse_dates=["starttime"])

TARGET_COLS = [c for c in sample_sub.columns if c != "Run No"]   # 2352 columns
print("train cases:", uc["Run No"].nunique(), "| test cases:", test_map["Run No"].nunique(),
      "| target cols:", len(TARGET_COLS))


train cases: 2922 | test cases: 725 | target cols: 2352


## 3. Feature engineering

In [3]:
# --- Feature engineering -----------------------------------------------------
# Problem is modelled as a LONG table: one row = (Run No, unit, hour) with
# target = committed in {0,1}.  The wide submission (2352 target cols) is
# unit-major (unit0 t0..t167, unit1 t0..t167, ...), so we can reshape directly.
# All features are built per (Run No, hour) [price/time/inflow/constraints] or
# per (Run No, reservoir) [water value/volume/inflow], then joined onto the long
# table. See GAMEPLAN.md and Report.ipynb -> Feature Engineering for rationale.
import re
from numpy.lib.stride_tricks import sliding_window_view

# tz-aware UTC -> naive so timestamps join cleanly
for _df in (price, inflow, minflow, minvol):
    if _df["date"].dt.tz is not None:
        _df["date"] = _df["date"].dt.tz_localize(None)
for _df in (volume, wvalue):
    if _df["date"].dt.tz is not None:
        _df["date"] = _df["date"].dt.tz_localize(None)
    _df["date"] = _df["date"].dt.normalize()

# ---- units / hours from the submission column order ----
_pat = re.compile(r"^result_committed_(.*)_t(\d+)$")
_units = [_pat.match(c).group(1) for c in TARGET_COLS]
UNITS  = list(dict.fromkeys(_units))          # 14 units, in wide order
U, H   = len(UNITS), 168
assert U * H == len(TARGET_COLS), "unexpected target layout"

# ---- topology: per-unit capacity/cost + plant->reservoir maps ----
import yaml
_topo_all = yaml.safe_load(open(EXTENDED / "Tokke_Vinje_topology.yaml"))
_topo = _topo_all["model"]
_gen, _res = _topo["generator"], _topo["reservoir"]
PLANT_OF = {u: u.rsplit("_G", 1)[0] for u in UNITS}
# Intake reservoir per plant, derived from the watercourse `connections` in the
# topology YAML: walk upstream from each plant through tunnels until a reservoir
# is reached. If a plant draws from several reservoirs (Songa: Songavatn and
# Bitdalsvatn) we use the largest one by max_vol.
_upstream, _downstream = {}, {}
for _c in _topo_all["connections"]:
    _upstream.setdefault(_c["to"], []).append((_c["from"], _c["from_type"]))
    _downstream.setdefault(_c["from"], []).append((_c["to"], _c["to_type"]))

def _intake_reservoirs(node, seen=None):
    seen = set() if seen is None else seen
    found = set()
    for src, src_type in _upstream.get(node, []):
        if src in seen:
            continue
        seen.add(src)
        if src_type == "reservoir":
            found.add(src)
        elif src_type == "tunnel":
            found |= _intake_reservoirs(src, seen)
    return found

def _outlet_reservoir(node, seen=None):
    """Follow a plant's discharge through rivers/tunnels to the next reservoir."""
    seen = set() if seen is None else seen
    for dst, dst_type in _downstream.get(node, []):
        if dst in seen:
            continue
        seen.add(dst)
        if dst_type == "reservoir":
            return dst
        if dst_type in ("river", "tunnel"):
            found = _outlet_reservoir(dst, seen)
            if found:
                return found
    return None

PLANT_RES, PLANT_OUT = {}, {}
for _p in dict.fromkeys(PLANT_OF.values()):
    _cands = _intake_reservoirs(_p)
    assert _cands, f"no intake reservoir found for plant {_p}"
    PLANT_RES[_p] = max(_cands, key=lambda r: _res[r]["max_vol"])
    PLANT_OUT[_p] = _outlet_reservoir(_p) or "(none)"   # Hogga: water leaves the system
print("Plant -> intake reservoir (from topology):", PLANT_RES)
print("Plant -> outlet reservoir (from topology):", PLANT_OUT)

_plants = list(dict.fromkeys(PLANT_OF.values()))
_n_at_plant = {p: list(PLANT_OF.values()).count(p) for p in _plants}
unit_static = pd.DataFrame({
    "unit_id":  np.arange(U),
    "unit":     UNITS,
    "plant_id": [_plants.index(PLANT_OF[u]) for u in UNITS],
    "reservoir":[PLANT_RES[PLANT_OF[u]] for u in UNITS],
    "outlet_res":[PLANT_OUT[PLANT_OF[u]] for u in UNITS],
    "p_min":    [_gen[u]["p_min"] for u in UNITS],
    "p_max":    [_gen[u]["p_max"] for u in UNITS],
    "p_nom":    [_gen[u]["p_nom"] for u in UNITS],
    "startcost":[_gen[u]["startcost_const"] for u in UNITS],
    "n_units_at_plant": [_n_at_plant[PLANT_OF[u]] for u in UNITS],
})

# ---- precomputed lookups ----
_price_s  = price.set_index("date")["Day-ahead price (EUR/MWh)"]
_infl_cols = [c for c in inflow.columns if c != "date"]
_infl     = inflow.set_index("date")[_infl_cols]
_infl_sum = _infl.sum(axis=1)
_wv  = wvalue.set_index("date"); _vol = volume.set_index("date")
_res_names = [c for c in wvalue.columns if c != "date"]
_minflow = minflow.set_index("date"); _minvol = minvol.set_index("date")
FWD_WINDOWS = (3, 6, 12)      # hours ahead for forward-looking price windows

def _fwd_stats(P, w):
    """Mean / max of the next w hours (incl. the current one) within each case.
    P: (cases, 168) price matrix; windows are truncated at the end of the week."""
    R_, H_ = P.shape
    cs  = np.concatenate([np.zeros((R_, 1)), np.cumsum(P, axis=1)], axis=1)
    idx = np.arange(H_); end = np.minimum(idx + w, H_)
    fmean = (cs[:, end] - cs[:, idx]) / (end - idx)
    Pp = np.pad(P, ((0, 0), (0, w - 1)), constant_values=np.nan)
    fmax = np.nanmax(sliding_window_view(Pp, w, axis=1), axis=2)
    return fmean, fmax

def build_rh(runs, starts):
    """Per (Run No, hour) features that do not depend on the unit."""
    R = len(runs)
    rh = pd.DataFrame({"Run No": np.repeat(runs, H),
                       "hour":   np.tile(np.arange(H), R)})
    rh["ts"] = np.repeat(np.asarray(starts), H) + pd.to_timedelta(rh["hour"], unit="h")
    rh["price"] = rh["ts"].map(_price_s)
    g = rh.groupby("Run No")["price"]
    rh["price_rank"]       = g.rank(pct=True)            # strongest single signal
    rh["price_wk_mean"]    = g.transform("mean")
    rh["price_wk_std"]     = g.transform("std")
    rh["price_minus_mean"] = rh["price"] - rh["price_wk_mean"]
    rh["price_z"]          = rh["price_minus_mean"] / rh["price_wk_std"].replace(0, np.nan)
    rh = rh.sort_values(["Run No", "hour"])
    rh["price_lag1"]  = g.shift(1)
    rh["price_lag24"] = g.shift(24)
    # Future-looking prices: SHOP optimises the whole week with every price known
    # in advance, so upcoming prices matter as much as past ones. Units also run in
    # blocks (min load + start costs), so windowed forward stats are informative.
    rh["price_lead1"]  = g.shift(-1)
    rh["price_lead24"] = g.shift(-24)
    P = rh["price"].to_numpy(dtype="float64").reshape(R, H)   # rows are (run, hour)-ordered
    for w in FWD_WINDOWS:
        fmean, fmax = _fwd_stats(P, w)
        rh[f"price_fwd_mean{w}"] = fmean.reshape(-1)
        rh[f"price_fwd_max{w}"]  = fmax.reshape(-1)
    rh["price_vs_fwd6"] = rh["price"] - rh["price_fwd_mean6"]
    # rank of this hour within its day of the case (cases start at 00:00 UTC)
    rh["day_in_case"]    = (rh["hour"] // 24).astype("int16")
    rh["price_rank_day"] = rh.groupby(["Run No", "day_in_case"])["price"].rank(pct=True)
    hod = rh["ts"].dt.hour; dow = rh["ts"].dt.day_of_week; doy = rh["ts"].dt.day_of_year
    rh["hour_of_day"] = hod.astype("int16"); rh["dow"] = dow.astype("int16")
    rh["month"] = rh["ts"].dt.month.astype("int16")
    rh["hour_in_horizon"] = rh["hour"].astype("int16")   # linear: horizon has a start/end
    for nm, x, p in [("hod", hod, 24), ("dow", dow, 7), ("doy", doy, 365.25)]:
        r = 2 * np.pi * x / p
        rh[f"{nm}_sin"] = np.sin(r); rh[f"{nm}_cos"] = np.cos(r)
    rh["inflow_total"] = rh["ts"].map(_infl_sum)
    # Hourly min-flow / min-volume constraints (previously unused). Only a few
    # rivers/reservoirs are ever restricted, so each gets its own column.
    for c in _minflow.columns:
        rh[f"minflow_{c}"] = rh["ts"].map(_minflow[c])
    for c in _minvol.columns:
        rh[f"minvol_{c}"] = rh["ts"].map(_minvol[c])
    return rh.drop(columns=["ts"])

def build_res_run(runs, starts):
    """Per (Run No, reservoir) water value / volume / inflow / min-volume features."""
    recs = []
    for run, st in zip(runs, starts):
        st = pd.Timestamp(st)
        d0 = st.normalize(); d1 = d0 + pd.Timedelta(days=7)
        wv0, wv1, v0 = _wv.asof(d0), _wv.asof(d1), _vol.asof(d0)
        infl_wk  = _infl.loc[st : st + pd.Timedelta(hours=H - 1)].mean()   # m3/s over the case
        infl_d1  = _infl.loc[st : st + pd.Timedelta(hours=23)].mean()      # first day
        mv0 = _minvol.asof(st)
        for rn in _res_names:
            recs.append((run, rn, wv0.get(rn, np.nan), wv1.get(rn, np.nan),
                         v0.get(rn, np.nan), _res.get(rn, {}).get("max_vol", np.nan),
                         infl_wk.get(rn, np.nan), infl_d1.get(rn, np.nan), mv0.get(rn, 0.0)))
    rr = pd.DataFrame(recs, columns=["Run No", "reservoir", "wv_start", "wv_end", "vol_start",
                                     "max_vol", "inflow_wk_mean", "inflow_d1_mean", "minvol_start"])
    rr["wv_trend"]  = rr["wv_end"] - rr["wv_start"]
    rr["fill_frac"] = rr["vol_start"] / rr["max_vol"]
    # weekly inflow volume (Mm3) relative to reservoir size -> spill-risk signal
    rr["inflow_wk_frac"]  = rr["inflow_wk_mean"] * 3600 * H / 1e6 / rr["max_vol"]
    rr["fill_plus_inflow"] = rr["fill_frac"] + rr["inflow_wk_frac"]
    # headroom above the minimum-volume requirement (0 for unrestricted reservoirs)
    rr["vol_margin_frac"] = (rr["vol_start"] - rr["minvol_start"]) / rr["max_vol"]
    return rr

def build_long(runs, starts, uc_sub=None):
    """Assemble the full long feature table for the given cases."""
    runs = np.asarray(runs); R = len(runs)
    rh = build_rh(runs, starts)
    rr = build_res_run(runs, starts)
    rr_sys = rr.groupby("Run No").agg(wv_sys_mean=("wv_start", "mean"),
                                      fill_sys_mean=("fill_frac", "mean"),
                                      inflow_wk_frac_sys=("inflow_wk_frac", "mean")).reset_index()
    # state of the reservoir each plant discharges INTO (downstream of the unit)
    rr_down = rr[["Run No", "reservoir", "fill_frac", "wv_end"]].rename(
        columns={"reservoir": "outlet_res", "fill_frac": "down_fill_frac", "wv_end": "down_wv_end"})
    long = pd.DataFrame({
        "Run No":  np.repeat(runs, U * H),
        "unit_id": np.tile(np.repeat(np.arange(U), H), R),
        "hour":    np.tile(np.tile(np.arange(H), U), R),
    })
    if uc_sub is not None:
        long["target"] = (uc_sub.set_index("Run No").loc[runs, TARGET_COLS]
                          .values.reshape(-1).astype(np.int8))
    long = (long.merge(unit_static, on="unit_id", how="left")
                .merge(rh, on=["Run No", "hour"], how="left")
                .merge(rr, on=["Run No", "reservoir"], how="left")
                .merge(rr_down, on=["Run No", "outlet_res"], how="left")
                .merge(rr_sys, on="Run No", how="left"))
    # core economic signal: price minus the unit's water value (opportunity cost)
    long["pwv_spread"]     = long["price"] - long["wv_start"].fillna(long["wv_sys_mean"])
    long["pwv_spread_end"] = long["price"] - long["wv_end"]
    long["price_wv_ratio"] = long["price"] / long["wv_end"].replace(0, np.nan)
    # Generating moves water from the intake to the outlet reservoir, so the net
    # opportunity cost is the water-value DROP between them (0 downstream of Hogga).
    long["wv_net"]         = long["wv_end"] - long["down_wv_end"].fillna(0)
    long["pwv_spread_net"] = long["price"] - long["wv_net"]
    # How many hours of this week are "in the money" for this unit, and how far
    # this hour's price rank sits above that cut-off.  Rows are (run, unit, hour)-
    # ordered (left merges keep order), so a reshape gives one row per (run, unit).
    _above = (long["pwv_spread_end"].to_numpy() > 0).reshape(-1, H).mean(axis=1)
    long["frac_hours_above_wv"] = np.repeat(_above, H)
    long["rank_above_cut"]      = long["price_rank"] - (1 - long["frac_hours_above_wv"])
    return long

# ---- build train & test tables -------------------------------------------
# For speed/memory you may subsample training hours (e.g. 2 -> every other hour).
SUBSAMPLE_HOURS = 1          # 1 = use all 168 hours
train_long = build_long(uc["Run No"].values, uc["starttime"].values, uc_sub=uc)
test_long  = build_long(test_map["Run No"].values, test_map["starttime"].values)
if SUBSAMPLE_HOURS > 1:
    train_long = train_long[train_long["hour"] % SUBSAMPLE_HOURS == 0].reset_index(drop=True)

# downcast to save memory on ~6.9M rows
for _df in (train_long, test_long):
    for c in _df.select_dtypes("float64").columns:
        _df[c] = _df[c].astype("float32")

print("train_long:", train_long.shape, "| test_long:", test_long.shape)
print("overall ON-rate:", round(float(train_long["target"].mean()), 4))

Plant -> intake reservoir (from topology): {'Hogga': 'Bandak', 'Tokke': 'Vinjevatn', 'Lio': 'Byrtevatn', 'Byrte': 'Botnedalsvatn', 'Vinje': 'Vaamarvatn', 'Haukeli': 'Vatjern', 'Songa': 'Songavatn', 'Kjela': 'Foersvatn', 'Vesle Kjela': 'Kjelavatn'}
Plant -> outlet reservoir (from topology): {'Hogga': '(none)', 'Tokke': 'Bandak', 'Lio': 'Bandak', 'Byrte': 'Byrtevatn', 'Vinje': 'Vinjevatn', 'Haukeli': 'Vinjevatn', 'Songa': 'Totak', 'Kjela': 'Hyljelihyl', 'Vesle Kjela': 'Foersvatn'}
train_long: (6872544, 76) | test_long: (1705200, 75)
overall ON-rate: 0.5164


## 4. Train model — XGBoost

In [4]:
# --- Train model: XGBoost ----------------------------------------------------
# Different predictor family than NB1 (LightGBM). Same long-table features, same
# leak-safe time split, same ON-rate priors. Nominal code columns (unit_id,
# plant_id, ...) use XGBoost's native categorical support. Hyperparameters were
# chosen to mirror NB1's capacity (max_depth 8 ~ num_leaves 255) and tuned on the
# time-split validation; see Report.ipynb -> Modeling for the search.
import xgboost as xgb
from sklearn.metrics import roc_auc_score

CAT_FEATURES = ["unit_id", "plant_id", "hour_of_day", "dow", "month"]
DROP = ["Run No", "unit", "reservoir", "outlet_res", "target"]

def add_on_rate_priors(fit_df, *apply_dfs):
    """Historical P(ON) per unit x hour-of-day and per unit x month, learned only
    from fit_df (leak-safe), mapped onto fit_df and every apply_df."""
    gm  = fit_df["target"].mean()
    phd = fit_df.groupby(["unit_id", "hour_of_day"])["target"].mean().rename("onrate_hod")
    pmo = fit_df.groupby(["unit_id", "month"])["target"].mean().rename("onrate_month")
    for d in (fit_df, *apply_dfs):
        d["onrate_hod"]   = d.merge(phd, on=["unit_id", "hour_of_day"], how="left")["onrate_hod"].fillna(gm).values
        d["onrate_month"] = d.merge(pmo, on=["unit_id", "month"], how="left")["onrate_month"].fillna(gm).values

def to_cat(df):
    for c in CAT_FEATURES:
        df[c] = df[c].astype("category")
    return df

XGB_PARAMS = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist",
                  max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  min_child_weight=50, reg_lambda=1.0, max_bin=255,
                  n_jobs=4, seed=RANDOM_SEED)
NUM_BOOST_ROUND = 1200

# -- honest validation on a time split (last ~15% of weeks) --
cut = int(uc["Run No"].quantile(0.85))
tr = to_cat(train_long[train_long["Run No"] <= cut].copy())
va = to_cat(train_long[train_long["Run No"] >  cut].copy())
add_on_rate_priors(tr, va)
FEATS = [c for c in tr.columns if c not in DROP]
dtr = xgb.DMatrix(tr[FEATS], label=tr["target"], enable_categorical=True)
dva = xgb.DMatrix(va[FEATS], label=va["target"], enable_categorical=True)
es  = xgb.callback.EarlyStopping(rounds=100, save_best=True, maximize=True,
                                 data_name="valid", metric_name="auc")
cv_model = xgb.train(XGB_PARAMS, dtr, num_boost_round=NUM_BOOST_ROUND,
                     evals=[(dtr, "train"), (dva, "valid")], callbacks=[es], verbose_eval=100)
va_auc = roc_auc_score(va["target"], cv_model.predict(dva))
BEST_ROUNDS = cv_model.best_iteration + 1
print(f"\nTime-split validation ROC-AUC: {va_auc:.5f}  (best_iter={BEST_ROUNDS})")

# -- final model: retrain on ALL train; priors recomputed on all train --
full = train_long.copy()
add_on_rate_priors(full)
_gm = full["target"].mean()
for col, keys in [("onrate_hod", ["unit_id", "hour_of_day"]),
                  ("onrate_month", ["unit_id", "month"])]:
    pri = full.groupby(keys)["target"].mean().rename(col)
    test_long[col] = test_long.merge(pri, on=keys, how="left")[col].fillna(_gm).values
FEATS = [c for c in full.columns if c not in DROP]
full = to_cat(full)
CAT_LEVELS = {c: full[c].cat.categories for c in CAT_FEATURES}   # align test codes
dall = xgb.DMatrix(full[FEATS], label=full["target"], enable_categorical=True)
model = xgb.train(XGB_PARAMS, dall, num_boost_round=BEST_ROUNDS, verbose_eval=200)
print("Final XGBoost model trained on", full.shape[0], "rows with", len(FEATS), "features.")


[0]	train-auc:0.95477	valid-auc:0.88815
[100]	train-auc:0.98794	valid-auc:0.96133
[200]	train-auc:0.99231	valid-auc:0.96242
[300]	train-auc:0.99439	valid-auc:0.96283
[400]	train-auc:0.99558	valid-auc:0.96244
[460]	train-auc:0.99612	valid-auc:0.96227

Time-split validation ROC-AUC: 0.96296  (best_iter=362)
Final XGBoost model trained on 6872544 rows with 73 features.


## 5. Predict test set & write `submission.csv`

In [5]:
# --- Predict test set & write submission.csv ---------------------------------
# Align test categorical levels to the training ones so codes match, then predict.
for c in CAT_FEATURES:
    test_long[c] = pd.Categorical(test_long[c], categories=CAT_LEVELS[c])
dtest = xgb.DMatrix(test_long[FEATS], enable_categorical=True)
test_long["p_on"] = model.predict(dtest).clip(0, 1)

# pivot long -> wide; column name = result_committed_<unit>_t<hour>
test_long["col"] = ("result_committed_" + test_long["unit"] + "_t"
                    + test_long["hour"].astype(str))
preds = (test_long.pivot_table(index="Run No", columns="col", values="p_on")
         .reindex(index=test_map["Run No"], columns=TARGET_COLS))

submission = preds.reset_index()
assert list(submission.columns) == ["Run No"] + TARGET_COLS
assert submission["Run No"].tolist() == sample_sub["Run No"].tolist()
assert submission[TARGET_COLS].isna().sum().sum() == 0, "missing predictions"
submission.to_csv("submission.csv", index=False)       # NB2's selected prediction
print("Wrote submission.csv", submission.shape)

# model-interpretation hook for Report.ipynb (XGBoost gain importance)
_gain = model.get_score(importance_type="gain")
imp = (pd.Series(_gain).rename("gain").reset_index().rename(columns={"index": "feature"})
       .sort_values("gain", ascending=False))
print("\nTop 15 features by gain:\n", imp.head(15).to_string(index=False))


Wrote submission.csv (725, 2353)

Top 15 features by gain:
             feature        gain
     pwv_spread_end 7588.076172
     price_wv_ratio 7045.946777
            max_vol 2204.844238
            unit_id 1611.885742
           plant_id 1586.395630
       onrate_month 1541.530640
     rank_above_cut 1254.138306
        down_wv_end  991.810486
   n_units_at_plant  781.934021
             wv_net  777.864136
         onrate_hod  744.562744
              price  654.069885
     inflow_wk_frac  641.187927
        wv_sys_mean  615.936096
minflow_b_Vinjevatn  611.398560


---
### Reproducibility notes
- Self-contained: runs offline from raw data in `DATA_DIR`; no external data; seed fixed.
- Same feature pipeline as Short notebook 1; only the learner differs (XGBoost vs LightGBM).
- Finishes well within the 12 h / 4-core / 32 GB budget.
- Writes `submission.csv` with columns identical to `sample_submission.csv`.
  *(If you run NB1 and NB2 back-to-back in the same folder, rename one output — both use this name.)*
